Hi there!
This is the code template for CW2 task2 of COMP34711 2025/26.

- <span style="color:red; font-size:1em">First of all, please rename the notebook into "{your_student_id}_CW2_task{your_task_number}.ipynb", for example "12345678_CW2_task2.ipynb".</span>

- In this template, we only provide the minimal structure for your coursework.
  
- Please carefully read and organize your code in the template we provided.

## Constants

In [1]:
#Please keep only necessary information in this cell.

#----------------------Please keep all following constants unchanged.----------------------------------------
NUM_ROWS_VALIDATION = 1031 # Number of rows in validation set
NUM_ROWS_TEST = 1053 # Number of rows in test set

#----------------------Please modify the following constants to fit your actual value.-----------------------
STUDENT_ID = '10879360'  # Replace with your actual 8-digits student ID
TRAINING_SET = './data/CW2_training_dataset.csv' # Replace with the actual path to your training dataset csv file
VALIDATION_SET = './data/CW2_validation_dataset.csv'  # Replace with the actual path to your validation dataset csv file
VALIDATION_SET_OUTPUT = f'./data/{STUDENT_ID}_CW2_task2_validation_results.csv'  # Replace with the actual path to your validation prediction csv file
TEST_SET_INPUT = './data/CW2_test_dataset.csv'  # Replace with the actual path to your test prediction csv file

#----------------------Your constants------------------------------------------------
# By adding more constants here, you can help improve the clarity and maintainability of your code and make the reviewing easier for TAs.

## Installations

In [2]:
# Install required packages for the coursework
# Uncomment and run the following lines if needed

!pip install pandas scikit-learn torch nltk --quiet
!pip install transformers accelerate datasets evaluate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


## Imports

In [ ]:
#Please keep all imports of your code cells in this cell

#---------------------Required imports----------------------
import pandas as pd
import re
import sys
import os.path
import csv
from sklearn.metrics import f1_score
#----------------------Your imports-------------------------
import numpy as np

# Pytorch tools
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler

# Transformer tools
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Fix seed for stable result
def set_seed(seed_value=378):
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    torch.cuda.manual_seed(seed_value)
    torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(378)

# Check a device that will run this code below
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda


## Start of your code cells

- The code cells provided below are demo code format for TAs to quickly locate your implementation.

- You have full right to freely add/delete/edit the titles and codes in the following cells.

- Please follow this genre order: "comedy, cult, flashback, historical, revenge, romantic, scifi, violence".

### Data Loading

In [17]:
# --------------------------------------------------------------------------------
# 1. Configuration & Tokenizer Initialization
# --------------------------------------------------------------------------------
MAX_LEN = 512
BATCH_SIZE = 32  # 32 for Tesla 4 GPU
MODEL_NAME = 'roberta-base'

print(f"Loading {MODEL_NAME} tokenizer~")
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# --------------------------------------------------------------------------------
# 2. Raw Data Loading (Clean Version)
# --------------------------------------------------------------------------------
# Load CSV files
df_train = pd.read_csv(TRAINING_SET)
df_validation = pd.read_csv(VALIDATION_SET)

print(f"✅ Raw Data Loaded (Train: {len(df_train)}, Validation: {len(df_validation)})")
print(f"   - Tokenizer initialized: {MODEL_NAME}")

Loading roberta-base tokenizer~
✅ Raw Data Loaded (Train: 7127, Validation: 1031)
   - Tokenizer initialized: roberta-base


### Tokenization

In [18]:
# --------------------------------------------------------------------------------
# 1. Preprocessing Function Definition (Optimized: Head+Tail + Separator)
# --------------------------------------------------------------------------------
def preprocess_data(df, tokenizer, max_len):
    """
    [Optimized] Preprocessing with Head+Tail Truncation and Separator.
    - Separates title and plot using the separator token (</s>).
    - Combines the beginning (Head) and end (Tail) of long texts to capture key information.
    """
    print(f"Tokenizing {len(df)} samples (Strategy: Head + Tail + Separator)~")

    input_ids_list = []
    attention_masks_list = []

    # RoBERTa Special Token IDs
    CLS_TOKEN = tokenizer.cls_token_id # <s>
    SEP_TOKEN = tokenizer.sep_token_id # </s>
    PAD_TOKEN = tokenizer.pad_token_id # <pad>

    # [Config] Head and Tail allocation (512 - 2 special tokens = 510 available)
    # Allocate 128 for Head (Title/Intro) and 382 for Tail (Ending)
    HEAD_LEN = 128
    TAIL_LEN = max_len - 2 - HEAD_LEN # 512 - 2 - 128 = 382

    titles = df['title'].astype(str).tolist()
    plots = df['plot_synopsis'].astype(str).tolist()

    for title, plot in zip(titles, plots):
        # 1. Tokenize title and plot separately (get IDs only, no special tokens)
        title_tokens = tokenizer.encode(title, add_special_tokens=False)
        plot_tokens = tokenizer.encode(plot, add_special_tokens=False)

        # 2. [Key Step 1] Insert Separator (Title + </s> + Plot)
        # Distinctly separates the title context from the plot context
        combined_tokens = title_tokens + [SEP_TOKEN] + plot_tokens

        # 3. [Key Step 2] Apply Head + Tail Truncation
        if len(combined_tokens) > max_len - 2:
            # If length exceeds limit, keep the beginning and end, discarding the middle
            combined_tokens = combined_tokens[:HEAD_LEN] + combined_tokens[-TAIL_LEN:]

        # 4. Add Special Tokens ([CLS] ~ [SEP]) and Padding
        # Final structure: [CLS] + (Title + SEP + Plot_Head + Plot_Tail) + [SEP]
        final_tokens = [CLS_TOKEN] + combined_tokens + [SEP_TOKEN]

        pad_len = max_len - len(final_tokens)
        input_ids = final_tokens + [PAD_TOKEN] * pad_len
        attention_mask = [1] * len(final_tokens) + [0] * pad_len

        input_ids_list.append(input_ids)
        attention_masks_list.append(attention_mask)

    # Convert lists to tensors
    input_ids = torch.tensor(input_ids_list, dtype=torch.long)
    attention_masks = torch.tensor(attention_masks_list, dtype=torch.long)

    # Handle Labels
    if len(df.columns) > 3:
        labels = torch.tensor(df.iloc[:, 3:].values, dtype=torch.float)
    else:
        labels = None

    return input_ids, attention_masks, labels

# --------------------------------------------------------------------------------
# 2. Preprocessing Execution & DataLoader Creation
# --------------------------------------------------------------------------------
# Execute Preprocessing
train_input_ids, train_attention_masks, train_labels = preprocess_data(df_train, tokenizer, MAX_LEN)
val_input_ids, val_attention_masks, val_labels = preprocess_data(df_validation, tokenizer, MAX_LEN)

print("✅ Preprocessing complete. Tensors created.")

# Create TensorDataset
train_dataset = TensorDataset(train_input_ids, train_attention_masks, train_labels)
val_dataset = TensorDataset(val_input_ids, val_attention_masks, val_labels)

# Create DataLoader (num_workers=2 recommended for Colab)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ DataLoaders Ready (Max Length: {MAX_LEN}, Batch Size: {BATCH_SIZE})")

Token indices sequence length is longer than the specified maximum sequence length for this model (941 > 512). Running this sequence through the model will result in indexing errors


Tokenizing 7127 samples (Strategy: Head + Tail + Separator)~
Tokenizing 1031 samples (Strategy: Head + Tail + Separator)~
✅ Preprocessing complete. Tensors created.
✅ DataLoaders Ready (Max Length: 512, Batch Size: 32)


### Model and Training

In [6]:
# --------------------------------------------------------------------------------
# 1. Asymmetric Loss Class (ASL)
# --------------------------------------------------------------------------------
class AsymmetricLoss(nn.Module):
    """
    Asymmetric Loss (ASL) to handle class imbalance in multi-label classification.
    It down-weights easy negatives and focuses on hard negatives.
    """
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, eps=1e-8):
        super(AsymmetricLoss, self).__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, x, y):
        targets = y
        xs_pos = torch.sigmoid(x)
        xs_neg = 1.0 - xs_pos

        # Margin shifting for negative samples
        if self.clip is not None and self.clip > 0:
            xs_neg = (xs_neg + self.clip).clamp(max=1)

        # Standard Cross-Entropy Terms
        los_pos = targets * torch.log(xs_pos.clamp(min=self.eps))
        los_neg = (1.0 - targets) * torch.log(xs_neg.clamp(min=self.eps))

        # Asymmetric Focusing Weights
        pos_weight = targets * (1.0 - xs_pos).pow(self.gamma_pos)
        neg_weight = (1.0 - targets) * xs_pos.pow(self.gamma_neg)

        loss = pos_weight * los_pos + neg_weight * los_neg
        return -loss.mean()

# --------------------------------------------------------------------------------
# 2. Model Definition: RoBERTa For Multi-Label Classification
# --------------------------------------------------------------------------------
class RoBERTaClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout_rate=0.1):
        super(RoBERTaClassifier, self).__init__()
        self.roberta = RobertaForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels,
            output_attentions=False,
            output_hidden_states=False
        )
        self.roberta.classifier.dropout = nn.Dropout(dropout_rate)

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.logits

# --------------------------------------------------------------------------------
# 3. Training & Evaluation Functions
# --------------------------------------------------------------------------------
def train_epoch(model, iterator, optimizer, scheduler, criterion, device, scaler):
    # Train model for one epoch using Mixed Precision.
    model.train()
    epoch_loss = 0

    for batch in iterator:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        labels = batch[2].to(device)

        optimizer.zero_grad()

        # Mixed Precision Training
        with autocast(device_type='cuda'):
            predictions = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(predictions, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion, device):
    # Evaluate model and find optimal thresholds per class.
    model.eval()
    epoch_loss = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for batch in iterator:
            input_ids = batch[0].to(device)
            attention_mask = batch[1].to(device)
            labels = batch[2].to(device)

            predictions = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()

            probs = torch.sigmoid(predictions)
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)

    # --- Find Optimal Thresholds (Class-wise) ---
    best_thresholds = []

    for i in range(8):
        best_f1_local = 0.0
        best_t_local = 0.5

        y_true = all_labels[:, i]
        y_prob = all_probs[:, i]

        # Search range: 0.30 to 0.70
        for t in np.arange(0.30, 0.71, 0.01):
            y_pred = (y_prob > t).astype(int)
            f1 = f1_score(y_true, y_pred, zero_division=0)

            if f1 > best_f1_local:
                best_f1_local = f1
                best_t_local = t

        best_thresholds.append(best_t_local)

    # Calculate final Weighted F1 using optimized thresholds
    final_preds = np.zeros_like(all_probs)
    for i in range(8):
        final_preds[:, i] = (all_probs[:, i] > best_thresholds[i]).astype(int)

    final_f1_weighted = f1_score(all_labels, final_preds, average='weighted', zero_division=0)

    return epoch_loss / len(iterator), final_f1_weighted, best_thresholds

# --------------------------------------------------------------------------------
# LLRD (Layer-wise Learning Rate Decay) Optimizer Group Function
# --------------------------------------------------------------------------------
def get_optimizer_grouped_parameters(model, learning_rate, weight_decay, layer_decay=0.95):
    """
    Creates parameter groups with layer-wise learning rate decay for RoBERTa.
    Lower layers get smaller learning rates.
    """
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = []

    # 1. Embeddings
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.roberta.roberta.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
        "weight_decay": weight_decay,
        "lr": learning_rate * (layer_decay ** 12)
    })
    optimizer_grouped_parameters.append({
        "params": [p for n, p in model.roberta.roberta.embeddings.named_parameters() if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
        "lr": learning_rate * (layer_decay ** 12)
    })

    # 2. Layers (Transformer Blocks)
    for layer_idx, layer in enumerate(model.roberta.roberta.encoder.layer):
        decay_power = 12 - (layer_idx + 1)
        lr_for_layer = learning_rate * (layer_decay ** decay_power)

        optimizer_grouped_parameters.append({
            "params": [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": weight_decay,
            "lr": lr_for_layer
        })
        optimizer_grouped_parameters.append({
            "params": [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
            "lr": lr_for_layer
        })

    # 3. Classifier Head
    head_params = [p for n, p in model.named_parameters() if "roberta.encoder" not in n and "roberta.embeddings" not in n]
    head_decay_params = [p for n, p in model.named_parameters() if "roberta.encoder" not in n and "roberta.embeddings" not in n and not any(nd in n for nd in no_decay)]
    head_no_decay_params = [p for n, p in model.named_parameters() if "roberta.encoder" not in n and "roberta.embeddings" not in n and any(nd in n for nd in no_decay)]

    optimizer_grouped_parameters.append({
        "params": head_decay_params,
        "weight_decay": weight_decay,
        "lr": learning_rate
    })
    optimizer_grouped_parameters.append({
        "params": head_no_decay_params,
        "weight_decay": 0.0,
        "lr": learning_rate
    })

    return optimizer_grouped_parameters

# --------------------------------------------------------------------------------
# 4. Main Execution Block
# --------------------------------------------------------------------------------

# 1. Hyperparameters
MODEL_NAME = 'roberta-base'
NUM_LABELS = 8
DROPOUT_RATE = 0.1
LEARNING_RATE = 2e-5
N_EPOCHS = 10

# 2. Initialize Model
print(f"Initializing {MODEL_NAME} Model~")
model = RoBERTaClassifier(MODEL_NAME, NUM_LABELS, DROPOUT_RATE)
model = model.to(device)

# 3. Optimizer with LLRD
grouped_parameters = get_optimizer_grouped_parameters(
    model,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    layer_decay=0.95
)
optimizer = torch.optim.AdamW(grouped_parameters)

# Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

scaler = GradScaler('cuda')

# 4. Loss Function (ASL)
criterion = AsymmetricLoss(gamma_neg=5, gamma_pos=1, clip=0.05).to(device)
print("✅ Loss Function set to ASL (gamma_neg=5).")

# 5. Training Loop
best_valid_f1 = 0.0
best_thresholds_list = [0.5] * 8

print(f"\nStarting RoBERTa Fine-tuning (LLRD + ASL + Class-wise Thresholds)~")
print("-" * 105)
print(f"{'Epoch':^6} | {'Train Loss':^10} | {'Val Loss':^10} | {'Val F1 (W)':^12} | {'Result':^10}")
print("-" * 105)

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, scaler)

    # Evaluation returns loss, F1, and optimal thresholds
    valid_loss, valid_f1_weighted, current_best_threshs = evaluate(model, val_loader, criterion, device)

    scheduler.step(valid_f1_weighted)

    if valid_f1_weighted > best_valid_f1:
        best_valid_f1 = valid_f1_weighted
        best_thresholds_list = current_best_threshs
        torch.save(model.state_dict(), f'{STUDENT_ID}_task2_best_model.pt')
        saved_msg = "🔥 Saved"
    else:
        saved_msg = ""

    print(f"{epoch+1:02}/{N_EPOCHS:02}  | {train_loss:.4f}     | {valid_loss:.4f}     | {valid_f1_weighted:.4f}       | {saved_msg}")

print("-" * 105)
print(f"✅ Training Complete.")
print(f"✅ Best Weighted F1 Score: {best_valid_f1:.4f}")
print(f"✅ Final Optimal Thresholds: {best_thresholds_list}")

Initializing roberta-base Model~


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Loss Function set to ASL (gamma_neg=5).

Starting RoBERTa Fine-tuning (LLRD + ASL + Class-wise Thresholds)~
---------------------------------------------------------------------------------------------------------
Epoch  | Train Loss |  Val Loss  |  Val F1 (W)  |   Result  
---------------------------------------------------------------------------------------------------------
01/10  | 0.0754     | 0.0686     | 0.5667       | 🔥 Saved
02/10  | 0.0668     | 0.0643     | 0.5995       | 🔥 Saved
03/10  | 0.0628     | 0.0624     | 0.6115       | 🔥 Saved
04/10  | 0.0599     | 0.0632     | 0.6172       | 🔥 Saved
05/10  | 0.0576     | 0.0617     | 0.6228       | 🔥 Saved
06/10  | 0.0546     | 0.0636     | 0.6253       | 🔥 Saved
07/10  | 0.0510     | 0.0677     | 0.6121       | 
08/10  | 0.0474     | 0.0702     | 0.6140       | 
09/10  | 0.0433     | 0.0787     | 0.6063       | 
10/10  | 0.0381     | 0.0809     | 0.6095       | 
-----------------------------------------------------------------

### Prediction

In [15]:
# --------------------------------------------------------------------------------
# Prediction Configuration
# --------------------------------------------------------------------------------
MODEL_PATH = f'{STUDENT_ID}_task2_best_model.pt'
OPTIMAL_THRESHOLDS = best_thresholds_list

# --------------------------------------------------------------------------------
# 2. Prediction Function (Head+Tail & Class-wise Thresholding)
# --------------------------------------------------------------------------------
def generate_prediction(input_file, output_file, thresholds_list):
    print(f"\n--- Loading Data from: {input_file} ---")
    df = pd.read_csv(input_file)

    # 1. Preprocessing (Reuse training logic)
    # Crucial: Must use the same Head+Tail+Separator strategy as training.
    print("Tokenizing data (Head + Tail + Separator Strategy)~")

    # The tokenizer must be defined in previous cells.
    # preprocess_data handles cases where labels are missing (returns None for labels).
    input_ids, attention_masks, _ = preprocess_data(df, tokenizer, MAX_LEN)

    # Create DataLoader
    dataset = TensorDataset(input_ids, attention_masks)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)

    # 2. Load Model
    print(f"Loading model from: {MODEL_PATH}")
    # Model architecture must match training configuration (Dropout 0.1)
    model = RoBERTaClassifier('roberta-base', 8, dropout_rate=0.1)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()

    # 3. Prediction Loop
    all_probs = []
    print("Starting Prediction~")

    with torch.no_grad():
        for batch in loader:
            b_input_ids = batch[0].to(device)
            b_input_mask = batch[1].to(device)

            # Forward pass
            outputs = model(b_input_ids, b_input_mask)
            probs = torch.sigmoid(outputs) # Convert logits to probabilities
            all_probs.extend(probs.cpu().numpy())

    all_probs = np.array(all_probs)

    # 4. Apply Class-wise Thresholds
    print(f"Applying Class-wise thresholds: {thresholds_list}")

    # Initialize integer array for binary predictions
    final_preds = np.zeros(all_probs.shape, dtype=int)

    for i in range(8):
        # Apply specific threshold for each genre i
        final_preds[:, i] = (all_probs[:, i] > thresholds_list[i]).astype(int)

    # 5. Save Results
    # Extract IDs
    ids = df.iloc[:, 0].values

    # Create DataFrame (ID + 8 Genres)
    df_output = pd.DataFrame(final_preds, columns=['comedy', 'cult', 'flashback', 'historical', 'revenge', 'romantic', 'scifi', 'violence'])

    # Ensure integer type
    df_output = df_output.astype(int)

    # Insert ID column at the beginning
    df_output.insert(0, 'ID', ids)

    # Save to CSV (No header, No index)
    df_output.to_csv(output_file, index=False, header=False)

    print(f"\n✅ Prediction Complete!")
    print(f"Saved to: {output_file}")
    print(f"Total Rows: {len(df_output)}")
    print(f"Example Row: {df_output.iloc[0].values}")

    return df_output

# 3. Execution
# Test with validation set first
# df_output = generate_prediction(VALIDATION_SET, VALIDATION_SET_OUTPUT, OPTIMAL_THRESHOLDS)

# Final Submission with TEST_SET
df_output = generate_prediction(TEST_SET_INPUT, f'{STUDENT_ID}_CW2_task2_results_prediction.csv', OPTIMAL_THRESHOLDS)


--- Loading Data from: ./data/CW2_test_dataset.csv ---
Tokenizing data (Head + Tail + Separator Strategy)~
Tokenizing 1053 samples (Strategy: Head + Tail + Separator)~


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading model from: 10879360_task2_best_model.pt
Starting Prediction~
Applying Class-wise thresholds: [np.float64(0.49000000000000016), np.float64(0.5900000000000003), np.float64(0.6000000000000003), np.float64(0.5600000000000003), np.float64(0.5500000000000003), np.float64(0.5600000000000003), np.float64(0.5000000000000002), np.float64(0.5900000000000003)]

✅ Prediction Complete!
Saved to: 10879360_CW2_task2_results_prediction.csv
Total Rows: 1053
Example Row: ['9484ac61-0e30-4799-9998-6f74f4cbb204' np.int64(0) np.int64(0)
 np.int64(0) np.int64(0) np.int64(1) np.int64(0) np.int64(0) np.int64(1)]


## End of your code cells

### Evaluation scripts

In [8]:
def read_data(submission_file_path, gold_standard_file_path):
    """
    Read submission and gold standard files.
    Extract student ID from filename.
    """
    # Try to find student ID from the filename (looks for 8 digit numbers)
    id_regex = r'\d{8}'

    user_id = re.findall(id_regex, submission_file_path)
    print("Found your ID: ", user_id)
    if user_id:
        user_id = user_id[0]
    else:
        user_id = 'Unknown'

    # Load submission CSV
    print(f"\nLoading submission file: {submission_file_path}")
    submission_df = pd.read_csv(submission_file_path, sep=',', header=None,
                                quoting=csv.QUOTE_NONE, encoding='utf-8')

    # Load gold standard CSV
    print(f"Loading gold standard file: {gold_standard_file_path}")
    gold_standard_df = pd.read_csv(gold_standard_file_path, header=None)

    # Remove columns 1 and 2 (keep only ID and labels)
    gold_standard_df = gold_standard_df.drop([1, 2], axis=1)
    # Skip header row
    gold_standard_df = gold_standard_df.iloc[1:]

    return submission_df, gold_standard_df, user_id


def match_and_prepare_data(submission_df, gold_standard_df, user_id):
    """
    Match submission rows with gold standard rows by ID.
    Prepare data for evaluation.
    """
    gold_standard_labels = []
    submission_labels = []
    missed_rows = []
    submission_df_copy = submission_df.copy()

    print(f"\nMatching submission with gold standard...")
    print(f"Gold standard rows: {len(gold_standard_df)}")
    print(f"Submission rows: {len(submission_df_copy)}")

    # Match each gold standard row with submission
    for index, row in gold_standard_df.iterrows():
        row = row.reset_index(drop=True)
        row_found = False
        row_id = row[0]

        # Extract gold standard labels
        row_labels = [int(row[i]) for i in range(1, len(row))]
        gold_standard_labels.append(row_labels)

        # Find corresponding submission row
        for sub_index, submission_row in submission_df_copy.iterrows():
            if submission_row[0].strip() == row_id.strip():
                try:
                    # Extract submission labels
                    submission_row_labels = [int(submission_row[i]) for i in range(1, len(submission_row))]
                except:
                    # Handle malformed labels (take first character if multi-digit)
                    submission_row_labels = [int(str(submission_row[i])[0]) for i in range(1, len(submission_row))]

                submission_labels.append(submission_row_labels)
                row_found = True
                submission_df_copy.drop(sub_index, inplace=True)
                break

        if not row_found:
            # If row is missing, add inverse labels (worst possible prediction)
            missed_rows.append(row_id)
            submission_labels.append([0 if label == 1 else 1 for label in row_labels])

    return gold_standard_labels, submission_labels, missed_rows


def evaluate_submission(gold_standard_labels, submission_labels):
    """
    Calculate weighted F1 score.
    """
    print(f"\nCalculating weighted F1 score...")

    # Calculate weighted F1 score (accounts for class imbalance)
    f1_weighted = f1_score(gold_standard_labels, submission_labels, average='weighted')

    return f1_weighted


def print_results(user_id, f1_weighted, missed_rows):
    """
    Print evaluation results to screen.
    """
    print("\n" + "="*70)
    print("YOUR SUBMISSION EVALUATION REPORT")
    print("="*70)

    # Alert if ID not found in filename
    if user_id == 'Unknown':
        print('WARNING: ID not found in filename!')
        print('   Please ensure your filename contains your 8-digit student ID.')
        print()

    print(f"Your ID: {user_id}")
    print()

    # Display F1 score with visual indicator
    print("EVALUATION RESULTS:")
    print(f"   Weighted F1 Score: {f1_weighted:.4f}")
    print()

    # Report missing rows
    if missed_rows:
        print(f"MISSING DATA ({len(missed_rows)} rows not found):")
        print("-" * 70)
        for i, row in enumerate(missed_rows[:10], 1):  # Show first 10
            print(f"    {i}. Row ID: {row}")
        if len(missed_rows) > 10:
            print(f"    ... and {len(missed_rows) - 10} more missing rows")
        print()
        print("TIP: Make sure your submission includes all required rows.")
        print("        Missing rows are penalized with worst possible predictions.")
    else:
        print("DATA COMPLETENESS: All expected rows found in your submission!")

    print()
    print("="*70)
    print()


def evaluate(submission_path, gold_standard_path):
    """
    Main function to run the submission evaluation script.
    """

    submission_file = submission_path
    gold_standard_file = gold_standard_path

    # Check if files exist
    if not os.path.exists(submission_file):
        print(f"Error: Your submission file '{submission_file}' not found!")
        print("Make sure the file path is correct and the file exists.")
        sys.exit(1)

    if not os.path.exists(gold_standard_file):
        print(f"Error: Gold standard file '{gold_standard_file}' not found!")
        print("Make sure you have the correct gold standard file.")
        sys.exit(1)

    try:
        # Step 1: Read data
        submission_df, gold_standard_df, user_id = read_data(submission_file, gold_standard_file)

        # Step 2: Match and prepare data
        gold_standard_labels, submission_labels, missed_rows = match_and_prepare_data(
            submission_df, gold_standard_df, user_id
        )

        # Step 3: Evaluate
        f1_weighted = evaluate_submission(gold_standard_labels, submission_labels)

        # Step 4: Print results
        print_results(user_id, f1_weighted, missed_rows)

    except Exception as e:
        print(f"Error during evaluation: {str(e)}")
        print("Please check that your files are in the correct CSV format.")
        print("Each row should contain: ID, label1, label2, label3, ...")
        import traceback
        traceback.print_exc()
        sys.exit(1)

### Evaluate the model on the validation dataset

In [14]:
# Please run the evaluation scripts cell above before running the mark_and_record

# Please make sure that output format is like following (no header row, no tilte and plot columns):
# 94834c61-0e30-4799-9998-6f74f6sbb204	0	1	0	0	1	0	0	0
# 559sdd28-b6a2-4662-ab55-a6678as26a56	0	0	0	0	0	0	1	0
# b71y3317-04cd-42f5-a380-d21dfasdbd36	0	0	0	0	1	0	0	0

evaluation_results = evaluate(VALIDATION_SET_OUTPUT, VALIDATION_SET)

Found your ID:  ['10879360']

Loading submission file: ./data/10879360_CW2_task2_validation_results.csv
Loading gold standard file: ./data/CW2_validation_dataset.csv

Matching submission with gold standard...
Gold standard rows: 1031
Submission rows: 1031

Calculating weighted F1 score...

YOUR SUBMISSION EVALUATION REPORT
Your ID: 10879360

EVALUATION RESULTS:
   Weighted F1 Score: 0.6253

DATA COMPLETENESS: All expected rows found in your submission!




### Save predictions to formatted file.

In [16]:
# Now please modify the code to format your output csv file.

# Please make sure that output format is like following (no header row, no title and plot columns):
# 94834c61-0e30-4799-9998-6f74f6sbb204	0	1	0	0	1	0	0	0
# 559sdd28-b6a2-4662-ab55-a6678as26a56	0	0	0	0	0	0	1	0
# b71y3317-04cd-42f5-a380-d21dfasdbd36	0	0	0	0	1	0	0	0

output_df = df_output  # Replace with your actual DataFrame or output
# For example, if you have a DataFrame named 'output_df', you can save it
assert isinstance(output_df, pd.DataFrame)
assert len(output_df) == NUM_ROWS_TEST, "Output length is not aligned with the testdata.csv."
assert len(output_df.columns) == 9, "Please make sure to follow the format above and keep only IDs and 8 columns of prediction."
output_df.to_csv(f'./{STUDENT_ID}_CW2_task2_results.csv', index=False, header=False)